<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 5 · Buscar y cruzar sin #N/D
**Excel Corporativo · 12 Claves** — BUSCARX, INDICE+COINCIDIR y su equivalente en pandas · ⏱ 60 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/05_busqueda_y_cruce.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-05)

**Al terminar podrás:**
1. Cruzar una exportación de ERP con el maestro de clientes.
2. Medir la tasa de cruce y clasificar por qué falla cada fila.
3. Normalizar llaves y separar los huérfanos que no se arreglan con fórmulas.
4. Evitar que un maestro con duplicados multiplique filas.

**Cómo usar este cuaderno.** Lee cada explicación y ejecuta la celda de código que sigue con **Shift + Enter**. Las celdas con un título y el código oculto (*Comprobar*, *Ver una solución*) se ejecutan igual; haz doble clic sobre ellas si quieres ver su código. Si algo falla, usa *Entorno de ejecución › Reiniciar y ejecutar todo*.

## Excel y Python, lado a lado

| En Excel | En pandas | Nota |
|---|---|---|
| `BUSCARX(llave; tClientes[id_cliente]; tClientes[segmento])` | `erp["llave"].map(clientes.set_index("id_cliente")["segmento"])` | Trae una columna |
| Varias columnas con BUSCARX | `erp.merge(clientes, how="left", ...)` | Trae varias columnas a la vez |
| `"No encontrado"` en el 4.º argumento | `indicator=True` → `left_only` | Marca lo que no cruzó |
| `ESPACIOS(...)` | `.str.strip()` | Python también quita el carácter 160 |
| Excel no distingue mayúsculas | pandas **sí** las distingue | Normaliza con `.str.upper()` |

In [ ]:
# @title Preparar el entorno (ejecuta esta celda primero)
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"   # si clonaste el repositorio, usa los archivos locales

def ruta(nombre):
    """Devuelve la ruta local del archivo si existe; si no, la URL de GitHub."""
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

def pesos(x):
    """Formato de pesos colombianos: $ 1.234.567"""
    return "$ " + f"{x:,.0f}".replace(",", ".")

def millones(x):
    return "$ " + f"{x/1e6:,.1f}".replace(",", "X").replace(".", ",").replace("X", ".") + " M"

def porc(v, signo=False):
    """Porcentaje con coma decimal: 18,9 %"""
    txt = f"{v:+.1%}" if signo else f"{v:.1%}"
    return txt.replace(".", ",").replace("%", " %")

def comprobar(respuesta, esperado, tolerancia=0.0, unidad=""):
    """Compara tu respuesta con el resultado esperado y te da pistas."""
    if respuesta is None:
        print("Aún no has escrito tu respuesta: reemplaza None por tu resultado y vuelve a ejecutar.")
        return
    if isinstance(esperado, str):
        ok = str(respuesta).strip().lower() == esperado.strip().lower()
    else:
        try:
            ok = abs(float(respuesta) - float(esperado)) <= tolerancia
        except (TypeError, ValueError):
            print("Tu respuesta debe ser un número."); return
    if ok:
        print(f"Correcto. {respuesta}{unidad} es el resultado esperado.")
    else:
        print(f"Todavía no. Revisa los filtros (¿quitaste las anuladas?) y el periodo. Tu respuesta: {respuesta}{unidad}")

# Plantilla de gráficos Dr. Bits: separador decimal coma y miles punto, como en Colombia
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=["#4A5AD8", "#0E9BB8", "#D0336F"],
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True),
    yaxis=dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True),
    separators=",.", hoverlabel=dict(font_size=13), margin=dict(t=70, l=10, r=20, b=40),
))
pio.templates.default = "plotly_white+drbits"
def dar_aire(fig, maximo, eje="x"):
    """Deja espacio a la derecha (o arriba) para que las etiquetas de las barras no se corten."""
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

MM = "Millones de pesos"   # en español «billón» es 10^12: evitamos la abreviatura B de Plotly
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
pd.set_option("display.max_columns", 30)
pd.set_option("styler.format.decimal", ",")      # tablas con estilo: 18,9 %
pd.set_option("styler.format.thousands", ".")
print("Entorno listo. Datos desde:", "archivos locales" if os.path.exists(BASE_LOCAL) else "GitHub")

## Parte A · Conocer las dos tablas
La exportación del ERP trae el código del cliente, pero no su **segmento** ni sus **días de crédito**, que viven en el maestro `clientes.csv`. La leemos toda como texto (`dtype=str`) para ver los datos exactamente como llegaron.

In [ ]:
erp = pd.read_csv(ruta("ventas_erp_sucio.csv"), dtype=str, keep_default_na=False)
clientes = pd.read_csv(ruta("clientes.csv"))
print("ERP:", erp.shape, "| Maestro:", clientes.shape)
erp.head(8)

En pantalla `C-0129` y `C-0129 ` se ven iguales. Para ver lo que hay de verdad, usa `repr` (muestra comillas y espacios) y mide la longitud. Un código válido tiene 6 caracteres. Es lo mismo que hacer `=LARGO(A2)` en Excel.

In [ ]:
erp["Cod Cliente"].str.len().value_counts().rename_axis("caracteres").to_frame("filas")

In [ ]:
raros = erp.loc[erp["Cod Cliente"].str.len() != 6, "Cod Cliente"]
raros.head(6).map(repr)

## Parte B · El cruce ingenuo
Primero crúzalas tal como vienen. `how="left"` conserva todas las filas del ERP (como una columna de BUSCARX al lado de tus datos) e `indicator=True` agrega la columna `_merge`, que dice si cada fila encontró pareja.

In [ ]:
maestro = clientes[["id_cliente", "segmento", "dias_credito"]]
ingenuo = erp.merge(maestro, left_on="Cod Cliente", right_on="id_cliente", how="left", indicator=True)
tasa_antes = (ingenuo["_merge"] == "both").mean()
print("Tasa de cruce sin limpiar:", porc(tasa_antes))
ingenuo["_merge"].value_counts().to_frame("filas")

### ¿Por qué falla cada fila?
Un #N/D no es un diagnóstico. Clasifica cada fallo según qué lo arreglaría: así sabes si el problema es de formato (se corrige) o de datos maestros (se reporta).

In [ ]:
ids = set(clientes["id_cliente"])

def motivo(cod):
    if cod in ids:                       return "Cruza tal como viene"
    if cod.strip() in ids:               return "Espacios"
    if cod.upper() in ids:               return "Minúsculas"
    if cod.strip().upper() in ids:       return "Espacios y minúsculas"
    return "No existe en el maestro"

ingenuo["motivo"] = ingenuo["Cod Cliente"].map(motivo)
motivos = ingenuo["motivo"].value_counts().rename_axis("motivo").reset_index(name="filas")
motivos

In [ ]:
colores = {"Cruza tal como viene": AZUL, "Espacios": TEAL, "Minúsculas": TEAL,
           "Espacios y minúsculas": TEAL, "No existe en el maestro": MAGENTA}
fig = px.bar(motivos.sort_values("filas"), x="filas", y="motivo", orientation="h", text="filas",
             color="motivo", color_discrete_map=colores,
             title="Qué pasa con cada fila del ERP al buscarla en el maestro")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(showlegend=False, yaxis_title="", xaxis_title="Filas", height=330)
fig.show()

> **Ojo con la diferencia entre herramientas.** En Excel, las filas con *Minúsculas* sí habrían cruzado: BUSCARX no distingue mayúsculas. En pandas no. Si migras un proceso de Excel a Python, el número de cruces puede cambiar sin que cambien los datos.

## Parte C · Normalizar la llave y cruzar bien
Crea una columna auxiliar con la llave limpia y no toques la original: así siempre puedes explicar qué transformaste. `validate="many_to_one"` le dice a pandas que el maestro debe tener cada código una sola vez; si no, se detiene con un error en vez de duplicar filas en silencio.

In [ ]:
erp["llave"] = erp["Cod Cliente"].str.strip().str.upper()
cruzado = erp.merge(maestro, left_on="llave", right_on="id_cliente", how="left",
                    indicator=True, validate="many_to_one")
tasa_despues = (cruzado["_merge"] == "both").mean()
print("Tasa de cruce:", porc(tasa_antes), "→", porc(tasa_despues))

In [ ]:
tasas = pd.DataFrame({"momento": ["Llave como viene", "Llave normalizada"],
                      "tasa": [tasa_antes, tasa_despues]})
fig = px.bar(tasas, x="momento", y="tasa", text=tasas["tasa"].map(lambda t: f"{t:.1%}".replace(".", ",")),
             color="momento", color_discrete_sequence=[GRIS, AZUL],
             title="Normalizar la llave antes de cruzar")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(showlegend=False, yaxis_tickformat=".0%", yaxis_range=[0, 1.1], xaxis_title="", yaxis_title="Filas cruzadas", height=360)
fig.show()

### Los huérfanos se reportan, no se fuerzan
Lo que sigue sin cruzar son códigos que **no existen** en el maestro: clientes creados en el ERP que nadie registró en el maestro. Ninguna fórmula lo arregla. El entregable correcto es una lista para quien administra los datos maestros.

In [ ]:
huerfanos = (cruzado[cruzado["_merge"] == "left_only"]
             .groupby("llave").agg(filas=("Factura", "size"), primera_factura=("Factura", "min"))
             .reset_index())
huerfanos.to_csv("huerfanos_para_maestro.csv", index=False)
print(len(huerfanos), "códigos huérfanos guardados en huerfanos_para_maestro.csv (panel de archivos de Colab)")
huerfanos.head()

### `map`: el BUSCARX de pandas
Si solo necesitas **una** columna, `map` con una serie indexada por la llave hace exactamente lo que hace BUSCARX: busca cada llave y devuelve el valor, o `NaN` si no la encuentra.

In [ ]:
segmento_por_id = clientes.set_index("id_cliente")["segmento"]
erp["segmento"] = erp["llave"].map(segmento_por_id).fillna("No encontrado")
erp[["Factura", "Cod Cliente", "llave", "segmento"]].head(8)

## Parte D · El riesgo del maestro duplicado
Si un código aparece dos veces en el maestro, BUSCARX devuelve la primera coincidencia **sin avisar**. En pandas el problema es otro: `merge` repite la fila del ERP una vez por cada coincidencia y los totales se inflan. Simulémoslo duplicando en el maestro al cliente que más compra.

In [ ]:
frecuente = erp["llave"].value_counts().index[0]          # el cliente con más líneas en el ERP
dup = clientes[clientes["id_cliente"] == frecuente].assign(razon_social="Registro duplicado S.A.S.")
print("Duplicamos en el maestro al cliente", frecuente)
maestro_dup = pd.concat([clientes, dup])[["id_cliente", "segmento", "dias_credito"]]

malo = erp.merge(maestro_dup, left_on="llave", right_on="id_cliente", how="left")
print("Filas del ERP:", len(erp), "→ después del merge:", len(malo))

try:
    erp.merge(maestro_dup, left_on="llave", right_on="id_cliente", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("\nCon validate, pandas se detiene a tiempo:\n", e)

En Excel la prevención equivalente es comprobar que la llave del maestro es única antes de cruzar: `=CONTAR.SI(tClientes[id_cliente];[@id_cliente])>1` en una columna auxiliar, o *Formato condicional › Valores duplicados*.

## Parte E · De vuelta al análisis
Con la llave resuelta, termina de preparar la exportación: elimina duplicados exactos (filas exportadas dos veces) y convierte el valor a número (es la clave 2). Luego responde la pregunta de negocio: ¿cuánto vende cada segmento?

In [ ]:
def a_pesos(texto):
    """'$ 1.234.567' → 1234567 ; '' → NaN"""
    limpio = texto.replace("$", "").replace(".", "").strip()
    return float(limpio) if limpio else np.nan

print("Líneas exportadas dos veces:", cruzado.duplicated(subset=["Linea"]).sum())
final = (cruzado.drop_duplicates(subset=["Linea"])
               .assign(valor=lambda d: d["Valor Neto"].map(a_pesos))
               .query("Estado != 'Anulada' and _merge == 'both'"))
por_segmento = final.groupby("segmento", as_index=False)["valor"].sum().sort_values("valor")
fig = px.bar(por_segmento, x=por_segmento["valor"] / 1e6, y="segmento", orientation="h", text=por_segmento["valor"].map(millones),
             color_discrete_sequence=[AZUL], title="Venta neta de la exportación por segmento (después del cruce)")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(xaxis_title=MM, yaxis_title="", height=330)
dar_aire(fig, por_segmento["valor"].max() / 1e6)
fig.show()

## Retos

### Reto 1
Después de normalizar la llave, ¿qué porcentaje de filas del ERP cruzó? Escríbelo como número con un decimal (por ejemplo, `97.3`).

*Pista:* `(cruzado['_merge'] == 'both').mean() * 100`.

In [ ]:
# Escribe tu código aquí y guarda el resultado en respuesta_1
respuesta_1 = None

In [ ]:
# @title Comprobar reto 1
comprobar(respuesta_1, round(tasa_despues * 100, 1), tolerancia=0.05, unidad=' %')

In [ ]:
# @title Ver una solución (inténtalo primero)
respuesta_1 = round((cruzado['_merge'] == 'both').mean() * 100, 1)
print(respuesta_1)

### Reto 2
¿Cuántos códigos de cliente **distintos** no existen en el maestro?

*Pista:* Filtra `left_only` y cuenta valores únicos de la llave.

In [ ]:
# Escribe tu código aquí y guarda el resultado en respuesta_2
respuesta_2 = None

In [ ]:
# @title Comprobar reto 2
comprobar(respuesta_2, cruzado.loc[cruzado['_merge'] == 'left_only', 'llave'].nunique(), unidad=' códigos')

In [ ]:
# @title Ver una solución (inténtalo primero)
respuesta_2 = cruzado.loc[cruzado['_merge'] == 'left_only', 'llave'].nunique()
print(respuesta_2)

### Reto 3
¿Cuántas filas de la exportación original están repetidas exactamente (exportadas dos veces)?

*Pista:* `erp_original.duplicated().sum()` sobre el archivo recién leído.

In [ ]:
# Escribe tu código aquí y guarda el resultado en respuesta_3
respuesta_3 = None

In [ ]:
# @title Comprobar reto 3
comprobar(respuesta_3, int(pd.read_csv(ruta('ventas_erp_sucio.csv'), dtype=str, keep_default_na=False).duplicated().sum()), unidad=' filas')

In [ ]:
# @title Ver una solución (inténtalo primero)
respuesta_3 = int(pd.read_csv(ruta('ventas_erp_sucio.csv'), dtype=str, keep_default_na=False).duplicated().sum())
print(respuesta_3)

## Apóyate en IA
Cuando un cruce falle, pide un diagnóstico, no una fórmula nueva:

```text
Estoy cruzando dos tablas en pandas con merge(how="left", indicator=True).
La tasa de cruce es [X %]. Estos son 10 valores de la llave que no cruzaron, mostrados con repr():
[pega aquí la lista]
Y estos son 5 valores válidos del maestro: [pega aquí]
Clasifica las causas probables (espacios, mayúsculas, caracteres invisibles, ceros a la izquierda,
códigos inexistentes) y dame el código para comprobar cada una. No modifiques la columna original.
```

> **Verifica siempre.** Ejecuta el código que te proponga sobre un caso cuyo resultado ya conoces. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Un cruce no termina cuando la fórmula funciona: termina cuando sabes **cuántas filas cruzaron y por qué fallaron las demás**.
* Normaliza la llave en una columna auxiliar: espacios, carácter 160 y, en Python, mayúsculas.
* Los huérfanos se reportan. Un maestro con llaves duplicadas se corrige antes de cruzar.

**Siguiente:** [Clave 7 · Tablas dinámicas que responden preguntas](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-07) · [Cuaderno 07](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/07_tablas_dinamicas.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>